# SIC k9 — Lovász theta

$SIC$ k=9, d=3, $\theta$-37.8, $\alpha$=36

In [ ]:
!pip install cvxpy pulp

In [ ]:
import numpy as np
import itertools
import cvxpy as cp
import pulp

def generate_rays(k):
    if k < 1 or int(k) != k:
        raise ValueError("k must be a positive integer")
    k = int(k)
    q = np.exp(2j * np.pi / k)

    vectors, labels = [], []

    vectors += [
        np.array([1, 0, 0], dtype=complex),
        np.array([0, 1, 0], dtype=complex),
        np.array([0, 0, 1], dtype=complex),
    ]
    labels += ["I:(1,0,0)", "I:(0,1,0)", "I:(0,0,1)"]

    for i in range(1, k + 1):
        qi = q ** i
        vectors += [
            np.array([1, -qi, 0], dtype=complex),
            np.array([1, 0, -qi], dtype=complex),
            np.array([0, 1, -qi], dtype=complex),
        ]
        labels += [f"II:i={i}:(1,-q^i,0)",
                   f"II:i={i}:(1,0,-q^i)",
                   f"II:i={i}:(0,1,-q^i)"]

    for i in range(1, k + 1):
        for j in range(1, k + 1):
            qi, qj = q ** i, q ** j
            vectors.append(np.array([1, qi, qj], dtype=complex))
            labels.append(f"III:(i={i},j={j}):(1,q^i,q^j)")

    expected_n = 3 + 3 * k + k * k
    assert len(vectors) == expected_n

    return vectors, labels

def is_orthogonal(u, v, tol=1e-7):
    nu = u / np.linalg.norm(u)
    nv = v / np.linalg.norm(v)
    return abs(np.vdot(nu, nv)) < tol


def build_adjacency(vectors, tol=1e-7):
    n = len(vectors)
    adj = np.zeros((n, n), dtype=int)
    for i, j in itertools.combinations(range(n), 2):
        if is_orthogonal(vectors[i], vectors[j], tol):
            adj[i, j] = adj[j, i] = 1
    return adj

def independence_number(adj):
    n = adj.shape[0]
    prob = pulp.LpProblem("max_independent_set", pulp.LpMaximize)
    x = {i: pulp.LpVariable(f"x_{i}", cat="Binary") for i in range(n)}
    prob += pulp.lpSum(x[i] for i in range(n))
    for i, j in itertools.combinations(range(n), 2):
        if adj[i, j] == 1:
            prob += x[i] + x[j] <= 1
    prob.solve(pulp.PULP_CBC_CMD(msg=False))
    return int(round(pulp.value(prob.objective)))

def lovasz_theta(adj, solver=cp.SCS, eps=1e-7, max_iters=200000, verbose=False):
    n = adj.shape[0]
    X = cp.Variable((n, n), symmetric=True)
    theta = cp.Variable()

    constraints = [X >> 0]
    for i in range(n):
        constraints.append(X[i, i] == theta - 1)
    for i, j in itertools.combinations(range(n), 2):
        if adj[i, j] == 0:
            constraints.append(X[i, j] == -1)

    prob = cp.Problem(cp.Minimize(theta), constraints)
    result = prob.solve(solver=solver, eps=eps, max_iters=max_iters,
                         verbose=verbose)
    return result

if __name__ == "__main__":
    k=6
    vectors, labels = generate_rays(k)
    n = len(vectors)
    print(f"k = {k}, n (number of rays) = {n}")

    adj = build_adjacency(vectors)
    print(f"Orthogonality graph edges = {int(adj.sum() / 2)}")

    alpha = independence_number(adj)
    print(f"Independence number alpha(G) = {alpha}  (known/expected: 36)")

    print("Solving SDP for Lovasz theta ...")
    theta_val = lovasz_theta(adj, eps=1e-7, max_iters=200000, verbose=False)
    print(f"\nLovasz theta(G) = {theta_val:.6f}")
    print(f"Sandwich check: alpha(G) = {alpha}  <=  theta(G) = {theta_val:.4f}")

k = 6, n (number of rays) = 57
Orthogonality graph edges = 174
Independence number alpha(G) = 18  (known/expected: 36)
Solving SDP for Lovasz theta ...

Lovasz theta(G) = 19.339816
Sandwich check: alpha(G) = 18  <=  theta(G) = 19.3398
